# 分群結果瀏覽器

## 結構
| Part | 功能 | 需改的變數 |
|---|---|---|
| **1** | 所有 cluster 統計總覽（讚數、留言數、預覽） | — |
| **2** | 單一 Cluster 深入瀏覽（全部留言依讚排序） | `CLUSTER_ID` |
| **3** | Cluster → 場景（centroid 法，Top-3 幕） | `CLUSTER_ID` |
| **4** | 留言 → 場景（每則留言個別對應，需先跑此 Cell） | `CLUSTER_ID` |
| **5** | **場景 → Clusters + 留言（反查：這一幕對應哪些群？）** | `EPISODE`, `SCENE_NUMBER` |

**建議流程**：從頭跑到 Part 4 的第一個 Cell（計算 comment-level 對應），之後可自由調整變數查看任意 cluster 或場景。

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

DATA = Path("data")

# 載入資料
comments  = pd.read_parquet(DATA / "comments_with_embedding.parquet")
clusters  = pd.read_parquet(DATA / "best_clusters.parquet")      # comment_id, text_clean, like_count, cluster_label
scenes    = pd.read_parquet(DATA / "scripts_with_embedding.parquet")  # 27 個場景
scene_map = pd.read_csv(DATA / "cluster_scene_mapping.csv", encoding="utf-8-sig")  # cluster → top-3 scenes

print(f"留言總數:  {len(clusters)}")
print(f"有效群數:  {clusters[clusters['cluster_label']!=-1]['cluster_label'].nunique()}")
print(f"Noise 數:  {(clusters['cluster_label']==-1).sum()}")
print(f"場景數:    {len(scenes)}")

留言總數:  10251
有效群數:  124
Noise 數:  3497
場景數:    27


---
## Part 1：分群總覽

所有 cluster 排列，顯示統計資料與最高讚留言預覽。

In [2]:
# 每個 cluster 的統計摘要
valid = clusters[clusters["cluster_label"] != -1]

summary = (
    valid.groupby("cluster_label")
    .agg(
        留言數   = ("comment_id", "count"),
        總讚數   = ("like_count", "sum"),
        最高讚   = ("like_count", "max"),
        平均讚   = ("like_count", "mean"),
    )
    .reset_index()
    .sort_values("總讚數", ascending=False)
)

# 加上每群最高讚的留言預覽
top_comment = (
    valid.sort_values("like_count", ascending=False)
    .groupby("cluster_label")
    .first()[["text_clean"]]
    .rename(columns={"text_clean": "最高讚留言（預覽）"})
)
top_comment["最高讚留言（預覽）"] = top_comment["最高讚留言（預覽）"].str[:60]

summary = summary.merge(top_comment, on="cluster_label")
summary["平均讚"] = summary["平均讚"].round(1)

pd.set_option("display.max_colwidth", 65)
pd.set_option("display.max_rows", 130)
summary

,cluster_label,留言數,總讚數,最高讚,平均讚,最高讚留言（預覽）
0,72,74,15854,7058,214.2,皇后:誰有孕我就害誰 華妃:誰得寵我就害誰 甄嬛:誰害我我就害誰 沈眉莊:誰害嬛兒我就害誰 安陵容:誰對我好我就害誰 果
1,20,88,15290,10621,173.8,小允子這種神隊友哪裡找? 會剪綵紙、會飛天扮鬼、會抓小偷、會打死敵人、會幫嬛嬛排除一切困難...
2,3,286,15273,3707,53.4,2023了 我還在恭迎熹妃娘娘回宮😚
3,113,157,14033,2615,89.4,"全程無尿點,太精彩了 過了這麼多年,甄嬛的扮相還是一樣讓人感覺非常美麗,一點都沒有因為時代前進而褪色TTT"
4,64,150,13157,3497,87.7,"我覺得寧嬪真的很完美,連最後自盡都是這麼的寧靜。 當甄嬛說:「謝謝你,你救了允禮的孩子。」的時候,寧嬪的第一個反應不是「"
5,67,96,11359,5886,118.3,寧貴人直接站上台 我直接噴笑 超像風紀股長
6,16,100,10396,7145,104.0,"皇上:我最期待的就是每次演吃丹藥的時候,因為都可以吃到巧克力球"
7,56,187,9848,2890,52.7,延禧攻略跟如㦤還有甄嬛 三部比一比 還是真心覺得追甄嬛好
8,17,36,9450,5098,262.5,"那年杏花微雨,你说你是果子狸,其实你是胖维尼"
9,84,73,9056,1972,124.1,"小知識:皇上死前拉劍帶,不是在掙扎,而是要告訴後人,他是冤死的。如果後人發現劍帶斷了,當時在皇帝身邊的人一律處死!在小說"


In [3]:
# c-TF-IDF 關鍵詞：計算所有 cluster 的代表字（跑一次，Part 2 使用）
from sklearn.feature_extraction.text import CountVectorizer

NGRAM_MIN    = 2
NGRAM_MAX    = 4
TOP_KEYWORDS = 8

all_cluster_ids = sorted(valid["cluster_label"].unique())
docs = [
    " ".join(valid[valid["cluster_label"] == cid]["text_clean"].dropna())
    for cid in all_cluster_ids
]

try:
    vec    = CountVectorizer(analyzer="char_wb", ngram_range=(NGRAM_MIN, NGRAM_MAX), min_df=2)
    tf_mat = vec.fit_transform(docs).toarray().astype(float)
    tf_norm = tf_mat / (tf_mat.sum(axis=1, keepdims=True) + 1e-9)
    idf     = np.log(len(docs) / ((tf_mat > 0).sum(axis=0) + 1)) + 1
    ctfidf  = tf_norm * idf
    names   = vec.get_feature_names_out()
    cluster_keywords = {
        cid: [names[j] for j in ctfidf[i].argsort()[-TOP_KEYWORDS:][::-1]]
        for i, cid in enumerate(all_cluster_ids)
    }
    print(f"關鍵詞計算完成，共 {len(cluster_keywords)} 個 cluster")
except ValueError as e:
    cluster_keywords = {cid: [] for cid in all_cluster_ids}
    print(f"關鍵詞計算失敗：{e}")

關鍵詞計算完成，共 124 個 cluster


---
## Part 1b：迷因排行榜

從四個角度找出最有迷因潛力的 cluster：
- **熱門迷因**：總讚高、群大、關鍵字清晰
- **潛力迷因**：平均讚高但群小（少數人在玩，尚未擴散）
- **Noise 中的孤狼**：未被分群但讚數高的單則留言（新梗萌芽信號）

In [ ]:
# ── 熱門迷因：總讚高 × 群大 × 平均讚穩定 ────────────────────────────
print("▌ 熱門迷因 Top 15（總讚 × 群規模）\n")
hot = summary.copy()
hot["平均讚"] = hot["平均讚"].round(1)
hot["kw"] = hot["cluster_label"].map(lambda c: " / ".join(cluster_keywords.get(c, [])[:4]))

pd.set_option("display.max_colwidth", 55)
display(
    hot.head(15)[["cluster_label", "留言數", "總讚數", "最高讚", "平均讚", "kw"]]
    .rename(columns={"kw": "代表字（前4）"})
    .reset_index(drop=True)
)

In [ ]:
# ── 潛力迷因：平均讚高但群小（小眾但高共鳴）────────────────────────
print("▌ 潛力迷因 Top 15（平均讚高、群規模 < 50）\n")
potential = summary[summary["留言數"] < 50].sort_values("平均讚", ascending=False).copy()
potential["kw"] = potential["cluster_label"].map(lambda c: " / ".join(cluster_keywords.get(c, [])[:4]))

display(
    potential.head(15)[["cluster_label", "留言數", "總讚數", "最高讚", "平均讚", "kw"]]
    .rename(columns={"kw": "代表字（前4）"})
    .reset_index(drop=True)
)

In [ ]:
# ── Noise 中的孤狼：未分群但讚數高（新梗萌芽信號）──────────────────
print("▌ Noise 中高讚留言 Top 20（cluster = -1，可能是尚未形成群體的新梗）\n")
noise = clusters[clusters["cluster_label"] == -1].sort_values("like_count", ascending=False)

pd.set_option("display.max_colwidth", 100)
display(
    noise.head(20)[["like_count", "text_clean"]]
    .rename(columns={"like_count": "讚數", "text_clean": "留言"})
    .reset_index(drop=True)
)

In [4]:
CLUSTER_ID = 3     # ← 改這裡（參考上面的 cluster_label 欄位）

grp  = clusters[clusters["cluster_label"] == CLUSTER_ID].sort_values("like_count", ascending=False)
stat = summary[summary["cluster_label"] == CLUSTER_ID].iloc[0]
kws  = cluster_keywords.get(CLUSTER_ID, [])

print(f"Cluster {CLUSTER_ID} — 留言數: {stat['留言數']}  總讚: {stat['總讚數']}  最高讚: {stat['最高讚']}")
print(f"代表字（c-TF-IDF）: {' ／ '.join(kws)}")
print(f"{'─'*70}")

pd.set_option("display.max_colwidth", 120)
grp[["like_count", "text_clean"]].reset_index(drop=True)

Cluster 3 — 留言數: 286  總讚: 15273  最高讚: 3707
代表字（c-TF-IDF）: 熹妃 ／ 回宮 ／ 迎熹 ／ 迎熹妃 ／ 妃回宮 ／ 妃回 ／ 恭迎熹 ／ 熹妃回宮
──────────────────────────────────────────────────────────────────────


,like_count,text_clean
0,3707,2023了 我還在恭迎熹妃娘娘回宮😚
1,2756,"2024年了,我還在迎接熹妃娘娘回宮❤"
2,2427,2022了 我還在恭迎熹娘娘回宮😌
3,1164,跳過片頭曲但一定要聽到‘’熹妃回宮‘’這四個字
4,1101,熹妃回宮~~ 用半副皇后儀仗然後大家都站著等她走路😆 難怪開始有人要加入甄嬛戰隊 畢竟這幾年皇上都是把她放在心上的 很喜歡嬛嬛後面跟皇后講話的妝😍
...,...,...
281,0,2022/8來向熹娘娘請安~
282,0,迎娘娘回宮到2023了
283,0,2033也要繼續迎熹妃娘娘!
284,0,熹妃回宮


In [5]:
CLUSTER_ID = 72     # ← 改這裡（參考上面的 cluster_label 欄位）

grp = clusters[clusters["cluster_label"] == CLUSTER_ID].sort_values("like_count", ascending=False)

stat = summary[summary["cluster_label"] == CLUSTER_ID].iloc[0]
print(f"Cluster {CLUSTER_ID} — 留言數: {stat['留言數']}  總讚: {stat['總讚數']}  最高讚: {stat['最高讚']}")
print(f"{'─'*70}")

pd.set_option("display.max_colwidth", 120)
grp[["like_count", "text_clean"]].reset_index(drop=True)

Cluster 72 — 留言數: 74  總讚: 15854  最高讚: 7058
──────────────────────────────────────────────────────────────────────


,like_count,text_clean
0,7058,皇后:誰有孕我就害誰 華妃:誰得寵我就害誰 甄嬛:誰害我我就害誰 沈眉莊:誰害嬛兒我就害誰 安陵容:誰對我好我就害誰 果郡王:誰穿綠衣我就撩誰 皇上:誰綠我我就寵誰
1,3195,"陵容的台詞安排得很妙,每句都像在幫甄環辯護,但每句都能讓祺貴人和皇后有新話題接。"
2,2393,從華妃 安陵容到皇后 發現每個甄嬛的敵人 死掉後 甄嬛的反應都不是很快樂 反而是一種感慨的感覺 也許她內心完全不希望她們真的死 而是被她們苦苦相逼不得不做出反擊
3,611,"都說甄嬛傳滴血驗親很經典,但我覺得最經典的是安陵容站在輸的一方,卻大勝全局。 1.安害死眉庒 2.讓甄嬛痛不欲生 3.讓愛面子的皇上丟臉 4.打擊皇后 5.讓瓜6進冷宮 6.貞嬪、康常在罰俸半年 安陵容溫太醫給眉庒看病那時,剛好到眉..."
4,545,安陵容的心機重其實也可以顯現她才是那個最喜好暗中窺探的人 要等所有人說完話才開口 並講出誰都不得罪的話🤦 其實我覺得她活著一定覺得很累._.
5,512,"沒錯,像4:35開始陵容的台詞,表面上在扮白臉,其實根本是在提醒祺貴人該cue靜白了。 這人真的好難對付啊好煩喔喔喔!!(莫名煩躁)"
6,404,我覺得安綾容最後跟甄嬛的對手戲也好看
7,236,安嬪其實是大堂之上最聰明狠毒的 表面上護著甄嬛實則落井下石 事蹟敗壞後又立刻建議皇上拔去淨白舌頭 自身撇得乾乾淨淨
8,230,安陵容發言的風格一直是這樣
9,124,"這也是皇后攻擊的路數之一。 想來安嬪拜在皇后門下,這幾年有潛心精進。"


---
## Part 3：Cluster → 場景對應（Cluster 層級）

**目前的做法：**
- 把一個 cluster 裡所有留言的 embedding 取**平均**（centroid）
- centroid 對 27 個場景的 embedding 算 cosine similarity
- 取 Top-3 場景

這個對應是「整群留言作為一個整體，最像哪一幕」，不是個別留言。

In [6]:
# 查看某個 cluster 對應到哪幾幕
CLUSTER_ID = 72     # ← 改這裡

rows = scene_map[scene_map["cluster_label"] == CLUSTER_ID].sort_values("scene_rank")
print(f"Cluster {CLUSTER_ID} 的 Top-3 對應場景：\n")
for _, r in rows.iterrows():
    cont = "(续)" if r["is_continuation"] else ""
    print(f"  #{int(r['scene_rank'])}  sim={r['similarity']:.3f}  {r['episode']} 第{int(r['scene_number'])}幕{cont}")
    print(f"     {r['scene_text'][:150]}")
    print()

Cluster 72 的 Top-3 對應場景：

  #1  sim=0.658  ep56 第894幕
     （景仁宫内室）
甄嬛：给皇后娘娘请安，恭祝娘娘凤体康健、千岁金安。
皇后：起来坐吧。
甄嬛：谢娘娘。
皇后：剪秋，看茶。今儿个也不是初一十五的大日子，熹妃这样早就来了。
甄嬛：本该昨日一回宫就来的，因而今日特来向娘娘请罪。
皇后：你有孕在身，又刚从甘露寺回来，是应该好好地歇息一下，反正日后都要见的，

  #2  sim=0.624  ep56 第899幕
     （寿康宫，眉嬛入）
眉庄：太后怎么先喝上药了，应该臣妾来喂您才是啊。
太后：你来得正好，除了你孙姑姑，也就你伺候得最叫哀家舒坦。
甄嬛：臣妾拜见太后，愿太后凤体康健、福泽万年。
太后：回来了？永寿宫住得还习惯？
甄嬛：永寿宫太过奢华，臣妾很是不安。
太后：虽然奢华，皇帝要宠着你，也不算什么。这药喝得

  #3  sim=0.607  ep76 第1184幕
     （景仁宫）
众：给皇太后请安。
甄嬛：景仁宫一切如旧，似乎还是昔年景象。
小允子：时移世易，人事早已不同。
甄嬛：皇后还似从前一样盯着那些鸽子看吗？
丫鬟：早些年是，先帝驾崩后皇后日夜痛哭，眼睛不大好了，便不再成天望着这些乱飞的鸽子。依太后娘娘的吩咐，这些鸽子老了就再养，总是要活蹦乱跳爱飞的那些。




In [7]:
# 所有 cluster 的主要對應場景（只看 rank=1）
top1 = scene_map[scene_map["scene_rank"] == 1].sort_values("likes_total", ascending=False)
top1 = top1[["cluster_label", "episode", "scene_number", "similarity", "size", "likes_total"]].copy()
top1["場景"] = top1["episode"] + " 第" + top1["scene_number"].astype(str) + "幕"
top1 = top1.drop(columns=["episode", "scene_number"]).rename(columns={"similarity": "sim", "size": "留言數", "likes_total": "總讚數"})
top1["sim"] = top1["sim"].round(3)
pd.set_option("display.max_rows", 130)
top1.reset_index(drop=True)

,cluster_label,sim,留言數,總讚數,場景
0,72,0.658,74,15854,ep56 第894幕
1,20,0.535,88,15290,ep76 第1178幕
2,3,0.638,286,15273,ep56 第891幕
3,113,0.603,157,14033,ep56 第894幕
4,64,0.634,150,13157,ep56 第894幕
5,67,0.631,96,11359,ep56 第894幕
6,16,0.487,100,10396,ep56 第896幕
7,56,0.597,187,9848,ep56 第894幕
8,17,0.482,36,9450,ep76 第1182幕
9,84,0.625,73,9056,ep76 第1179幕


---
## Part 4：留言層級對應（Comment → 場景）

**目前的做法 vs. 這裡的做法：**
- Part 3 是 cluster 整體對應
- 這裡是**每則留言個別**對它的 embedding 和 27 個場景算 cosine similarity，取 Top-1

計算量較大（10k 留言 × 27 場景），但只算一次。

In [8]:
# 計算每則留言的 Top-1 場景（只在 valid cluster 的留言中算）
valid_comments = clusters[clusters["cluster_label"] != -1].merge(
    comments[["comment_id", "embedding"]], on="comment_id", how="left"
)

comment_matrix = np.stack(valid_comments["embedding"].values)   # (n, 4096)
scene_matrix   = np.stack(scenes["embedding"].values)           # (27, 4096)
scene_matrix   = scene_matrix / (np.linalg.norm(scene_matrix, axis=1, keepdims=True) + 1e-9)

# cosine sim — batch 計算避免記憶體爆炸
BATCH = 500
top1_scene_idx = []
top1_scene_sim = []
for i in range(0, len(comment_matrix), BATCH):
    batch = comment_matrix[i:i+BATCH]
    batch = batch / (np.linalg.norm(batch, axis=1, keepdims=True) + 1e-9)
    sim = batch @ scene_matrix.T   # (batch, 27)
    top1_scene_idx.extend(sim.argmax(axis=1).tolist())
    top1_scene_sim.extend(sim.max(axis=1).tolist())

valid_comments = valid_comments.copy()
valid_comments["scene_idx"]  = top1_scene_idx
valid_comments["scene_sim"]  = top1_scene_sim
valid_comments["scene_ep"]   = valid_comments["scene_idx"].map(lambda i: scenes.iloc[i]["episode"])
valid_comments["scene_num"]  = valid_comments["scene_idx"].map(lambda i: scenes.iloc[i]["scene_number"])
valid_comments["scene_label"] = valid_comments["scene_ep"] + " 第" + valid_comments["scene_num"].astype(str) + "幕"

print(f"計算完成：{len(valid_comments)} 則留言")
print(f"\n留言最多對應到的場景 Top-10：")
print(valid_comments["scene_label"].value_counts().head(10).to_string())

計算完成：6754 則留言

留言最多對應到的場景 Top-10：
scene_label
ep76 第1182幕    2364
ep56 第894幕     1298
ep76 第1179幕     925
ep56 第897幕      381
ep63 第996幕      321
ep56 第891幕      247
ep56 第896幕      179
ep76 第1178幕     164
ep56 第899幕      163
ep76 第1184幕     140


In [9]:
# 查看某個 cluster 的留言分別對應到哪幾幕
CLUSTER_ID = 3     # ← 改這裡

grp = valid_comments[valid_comments["cluster_label"] == CLUSTER_ID].sort_values("like_count", ascending=False)

print(f"Cluster {CLUSTER_ID} 的留言 → 個別場景對應：\n")
print(f"{'讚數':>6}  {'場景':15}  {'sim':5}  留言")
print("─" * 80)
for _, r in grp.head(20).iterrows():
    print(f"{int(r['like_count']):>6}  {r['scene_label']:15}  {r['scene_sim']:.3f}  {str(r['text_clean'])[:50]}")

Cluster 3 的留言 → 個別場景對應：

    讚數  場景               sim    留言
────────────────────────────────────────────────────────────────────────────────
  3707  ep56 第891幕       0.565  2023了 我還在恭迎熹妃娘娘回宮😚
  2756  ep56 第891幕       0.606  2024年了,我還在迎接熹妃娘娘回宮❤
  2427  ep56 第891幕       0.558  2022了 我還在恭迎熹娘娘回宮😌
  1164  ep56 第894幕       0.540  跳過片頭曲但一定要聽到‘’熹妃回宮‘’這四個字
  1101  ep56 第894幕       0.623  熹妃回宮~~ 用半副皇后儀仗然後大家都站著等她走路😆 難怪開始有人要加入甄嬛戰隊 畢竟這幾年皇上都是
   971  ep56 第894幕       0.550  5:00 熹妃是在諷刺安嬪從她出宮到回宮都一樣還是嬪位 所以安嬪臉色突然變難看
   351  ep56 第891幕       0.487  熹妃回宮的造型又美又霸氣!!!!
   348  ep56 第897幕       0.445  Aki CCC 這部戲緊緻的地方在這,從開始的淡妝樸素 到復仇的熹妃 那眉毛衣服妝容 真的是厲害細緻
   344  ep56 第894幕       0.529  2023快結束了我在恭迎熹妃回宮😂
   342  ep56 第891幕       0.573  2024已經過一半了 我還在恭迎熹貴妃回宮😍
   232  ep56 第891幕       0.571  2024一開始我就在恭迎熹妃娘娘回宮😍
   179  ep56 第894幕       0.499  太厲害的妝容了,熹貴妃哭泣成那樣,睫毛膏眼線都沒暈開,真想知道是何牌子(離題了^^")
   129  ep56 第891幕       0.589  2024了我還在恭迎熹妃娘娘回宮
    92  ep56 第891幕       0.536  2023快結束了 我還在恭迎熹妃娘娘回宮🥰
    78  ep56 第894幕       0.

In [10]:
# 某個 cluster 的場景分布（哪幾幕留言最多）
CLUSTER_ID = 3     # ← 改這裡

grp = valid_comments[valid_comments["cluster_label"] == CLUSTER_ID]
print(f"Cluster {CLUSTER_ID} 的留言場景分布：")
print(grp["scene_label"].value_counts().to_string())

Cluster 3 的留言場景分布：
scene_label
ep56 第891幕     152
ep56 第897幕      66
ep56 第894幕      64
ep76 第1179幕      3
ep56 第899幕       1


In [11]:
# 比較 cluster centroid 法 vs 個別留言法的差異
CLUSTER_ID = 3     # ← 改這裡

centroid_top1 = scene_map[(scene_map["cluster_label"] == CLUSTER_ID) & (scene_map["scene_rank"] == 1)].iloc[0]
comment_top1  = valid_comments[valid_comments["cluster_label"] == CLUSTER_ID]["scene_label"].value_counts().idxmax()

print(f"Cluster {CLUSTER_ID}")
print(f"  Centroid 法（整群平均）→ {centroid_top1['episode']} 第{int(centroid_top1['scene_number'])}幕  sim={centroid_top1['similarity']:.3f}")
print(f"  個別留言法（多數決）   → {comment_top1}")

Cluster 3
  Centroid 法（整群平均）→ ep56 第891幕  sim=0.638
  個別留言法（多數決）   → ep56 第891幕


---
## Part 5：場景瀏覽器（Scene → Clusters + 留言）

從場景的角度反查：**這一幕吸引了哪些 cluster？留言在說什麼？**

先跑下面第一個 Cell 確認所有可選的場景，再修改 `EPISODE` 和 `SCENE_NUMBER` 深入查看。

> 需要先跑完 Part 4 才能使用（依賴 `valid_comments`）。

In [12]:
# 所有可選場景一覽（含統計），依總讚數排序
scene_overview = (
    valid_comments
    .groupby(["scene_ep", "scene_num"])
    .agg(
        對應留言數 = ("comment_id", "count"),
        總讚數     = ("like_count", "sum"),
        最高讚     = ("like_count", "max"),
        平均sim    = ("scene_sim", "mean"),
    )
    .reset_index()
    .rename(columns={"scene_ep": "episode", "scene_num": "scene_number"})
)

# 加上該場景被幾個 cluster centroid 指向（rank=1）
centroid_count = (
    scene_map[scene_map["scene_rank"] == 1]
    .groupby(["episode", "scene_number"])
    .size()
    .reset_index(name="Cluster數_centroid")
)
scene_overview = scene_overview.merge(centroid_count, on=["episode", "scene_number"], how="left")
scene_overview["Cluster數_centroid"] = scene_overview["Cluster數_centroid"].fillna(0).astype(int)
scene_overview["平均sim"] = scene_overview["平均sim"].round(3)
scene_overview = scene_overview.sort_values("總讚數", ascending=False).reset_index(drop=True)

pd.set_option("display.max_rows", 30)
print("可選場景（依總讚數排序）：")
scene_overview

可選場景（依總讚數排序）：


,episode,scene_number,對應留言數,總讚數,最高讚,平均sim,Cluster數_centroid
0,ep56,894,1298,111486,7058,0.496,32
1,ep76,1179,925,52533,7145,0.455,15
2,ep76,1182,2364,48376,6445,0.366,52
3,ep76,1178,164,22709,10621,0.509,3
4,ep63,996,321,19807,8144,0.473,3
5,ep56,891,247,13364,3707,0.555,3
6,ep56,899,163,12791,2909,0.517,1
7,ep76,1184,140,10535,3075,0.533,1
8,ep56,898,71,9536,1965,0.533,1
9,ep63,1002,111,9535,4336,0.522,3


In [13]:
# ── 修改這兩個變數來選擇場景 ──────────────────────────────────────────
EPISODE      = "ep56"   # 選項：\"ep56\" / \"ep63\" / \"ep76\"
SCENE_NUMBER = 894      # 參考上方表格的 scene_number
# ─────────────────────────────────────────────────────────────────────

scene_row = scenes[(scenes["episode"] == EPISODE) & (scenes["scene_number"] == SCENE_NUMBER)]
if scene_row.empty:
    print(f"找不到 {EPISODE} 第{SCENE_NUMBER}幕，請確認上方表格")
else:
    sr = scene_row.iloc[0]
    scene_label = f"{EPISODE} 第{SCENE_NUMBER}幕"
    cont_tag = "（续）" if sr["is_continuation"] else ""

    matched    = valid_comments[valid_comments["scene_label"] == scene_label]
    c_clusters = scene_map[
        (scene_map["episode"] == EPISODE) &
        (scene_map["scene_number"] == SCENE_NUMBER) &
        (scene_map["scene_rank"] == 1)
    ]

    print(f"{'='*65}")
    print(f"{EPISODE} 第{SCENE_NUMBER}幕{cont_tag}")
    print(f"{'='*65}")
    print(f"  對應留言數（個別留言法）    : {len(matched)}")
    print(f"  對應 Cluster 數（centroid）: {len(c_clusters)}")
    print(f"  留言總讚數                  : {matched['like_count'].sum():,}")
    print(f"  留言最高讚                  : {matched['like_count'].max():,}")
    print(f"  平均 sim                    : {matched['scene_sim'].mean():.3f}")
    print(f"\n── 場景原文 {'─'*47}")
    print(sr["text"])

ep56 第894幕
  對應留言數（個別留言法）    : 1298
  對應 Cluster 數（centroid）: 32
  留言總讚數                  : 111,486
  留言最高讚                  : 7,058
  平均 sim                    : 0.496

── 場景原文 ───────────────────────────────────────────────
（景仁宫内室）
甄嬛：给皇后娘娘请安，恭祝娘娘凤体康健、千岁金安。
皇后：起来坐吧。
甄嬛：谢娘娘。
皇后：剪秋，看茶。今儿个也不是初一十五的大日子，熹妃这样早就来了。
甄嬛：本该昨日一回宫就来的，因而今日特来向娘娘请罪。
皇后：你有孕在身，又刚从甘露寺回来，是应该好好地歇息一下，反正日后都要见的，请安也不急在一时。
甄嬛：皇后关怀，臣妾也不敢太放肆。臣妾在外，不敢忘了皇后恩德，因此日日祝祷，奉了佛珠在佛前开光，希望有朝一日可以奉送给娘娘，保佑娘娘岁岁安康。
皇后：熹妃有心。其实东西还是其次，最要紧的是妹妹一番聪慧，知道终有一日还能与本宫相见。
绘春：请皇后娘娘替花。
（甄嬛为皇后簪花）
皇后：熹妃从前服侍本宫替花的规矩倒一点都没错。
甄嬛：服侍娘娘是应当的，臣妾不敢忘了规矩。
皇后：一晃数年，瞧着熹妃的样子不改分毫，倒似更见风韵了，当真连岁月匆匆都格外疼惜熹妃，全不似本宫人老珠黄了。
甄嬛：皇后娘娘母仪天下，如这牡丹雍容华贵，不知娘娘为何出此伤感之语？
皇后：牡丹又如何，凭它什么花都会有开有谢，只是早晚而已。
剪秋：娘娘，各宫嫔妃来请安，已在外头等候了。
皇后：熹妃，一同去吧。
（正殿）
皇后：如今熹妃回来了，敬妃你也要多带着胧月公主去熹妃宫里走走，到底熹妃是胧月的生母，等熹妃生产之后胧月公主也该送回永寿宫去，你这个养娘再亲，到底也比不上人家亲娘。
敬妃：臣妾遵旨。
皇后：先坐下吧。对了，宁贵人呢？为什么今日没来呢？
陵容：回娘娘，宁贵人一早起说不舒服，是而不能向皇后请安了。
皇后：既然她不舒服，就叫太医好好照看着吧。熹妃啊，你有孕，身子要好好养着才是，要多歇息、少走动，能免的礼数就免了吧。
甄嬛：谢娘娘。
陵容：熹妃有了四阿哥和公主，这样的福气哪里是人人都能学来的？
皇后：好了，今天就这样吧，你们都散了吧。
众人：臣妾

In [14]:
# 對應到此場景的 Clusters（centroid 法，rank=1），依總讚數排序
c_table = (
    scene_map[
        (scene_map["episode"] == EPISODE) &
        (scene_map["scene_number"] == SCENE_NUMBER) &
        (scene_map["scene_rank"] == 1)
    ]
    .merge(
        summary[["cluster_label", "留言數", "總讚數", "最高讚", "最高讚留言（預覽）"]],
        on="cluster_label", how="left"
    )
    [["cluster_label", "similarity", "留言數", "總讚數", "最高讚", "最高讚留言（預覽）"]]
    .sort_values("總讚數", ascending=False)
    .rename(columns={"similarity": "sim"})
    .reset_index(drop=True)
)
c_table["sim"] = c_table["sim"].round(3)

print(f"{EPISODE} 第{SCENE_NUMBER}幕 ← {len(c_table)} 個 Cluster 的 centroid 最近此幕")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)
c_table

ep56 第894幕 ← 32 個 Cluster 的 centroid 最近此幕


,cluster_label,sim,留言數,總讚數,最高讚,最高讚留言（預覽）
0,72,0.658,74,15854,7058,皇后:誰有孕我就害誰 華妃:誰得寵我就害誰 甄嬛:誰害我我就害誰 沈眉莊:誰害嬛兒我就害誰 安陵容:誰對我好我就...
1,113,0.603,157,14033,2615,"全程無尿點,太精彩了 過了這麼多年,甄嬛的扮相還是一樣讓人感覺非常美麗,一點都沒有因為時代前進而褪色TTT"
2,64,0.634,150,13157,3497,"我覺得寧嬪真的很完美,連最後自盡都是這麼的寧靜。 當甄嬛說:「謝謝你,你救了允禮的孩子。」的時候,寧嬪的第一個反..."
3,67,0.631,96,11359,5886,寧貴人直接站上台 我直接噴笑 超像風紀股長
4,56,0.597,187,9848,2890,延禧攻略跟如㦤還有甄嬛 三部比一比 還是真心覺得追甄嬛好
5,79,0.564,57,6751,4484,"「那年杏花微雨,你說你是果郡王,也許從一開始,便都是錯的。」 這句太經典了!貫穿整部劇的關鍵"
6,62,0.604,61,6623,3714,"玉嬈有純元的面貌,年世蘭的桀驁不馴,甄嬛的氣韻和家承一脈教養,依老色鬼的角度看真是完美啊"
7,90,0.660,93,5266,2574,"「哀家能有今日,全靠皇后您一手指點歷練,自然感恩戴德。」 這句嗆爆🤣"
8,99,0.672,30,4817,2909,盤點一下幾場百看不厭的精彩戲碼: 1.甄嬛第一次小產 2.華妃與甄嬛在冷宮的對手戲 3.滴血驗親 4.眉莊之死 ...
9,75,0.660,78,4727,1194,"祺貴人:論門第、樣貌,哪一點比不上你 大眾想:腦子 皇后:若是姐姐在的話,一定相信臣妾是親白的 姐姐的靈魂表示:..."


In [15]:
# 對應到此場景的所有留言（個別留言法），依讚數排序
TOP_COMMENTS = 50   # 最多顯示幾則，改 None 顯示全部

matched_comments = (
    valid_comments[valid_comments["scene_label"] == scene_label]
    .sort_values("like_count", ascending=False)
    [["like_count", "cluster_label", "scene_sim", "text_clean"]]
    .rename(columns={"like_count": "讚數", "cluster_label": "cluster", "scene_sim": "sim", "text_clean": "留言"})
    .reset_index(drop=True)
)
matched_comments["sim"] = matched_comments["sim"].round(3)

print(f"{EPISODE} 第{SCENE_NUMBER}幕 ← {len(matched_comments)} 則留言（個別留言法），顯示前 {TOP_COMMENTS or len(matched_comments)} 則")
pd.set_option("display.max_colwidth", 100)
matched_comments.head(TOP_COMMENTS)

ep56 第894幕 ← 1298 則留言（個別留言法），顯示前 50 則


,讚數,cluster,sim,留言
0,7058,72,0.587,皇后:誰有孕我就害誰 華妃:誰得寵我就害誰 甄嬛:誰害我我就害誰 沈眉莊:誰害嬛兒我就害誰 安陵容:誰對我好我就害誰 果郡王:誰穿綠衣我就撩誰 皇上:誰綠我我就寵誰
1,5886,67,0.432,寧貴人直接站上台 我直接噴笑 超像風紀股長
2,5098,17,0.407,"那年杏花微雨,你说你是果子狸,其实你是胖维尼"
3,3714,62,0.526,"玉嬈有純元的面貌,年世蘭的桀驁不馴,甄嬛的氣韻和家承一脈教養,依老色鬼的角度看真是完美啊"
4,3662,25,0.532,每次看到這集都一直在想...熹貴妃的口紅色號
5,3497,64,0.563,"我覺得寧嬪真的很完美,連最後自盡都是這麼的寧靜。 當甄嬛說:「謝謝你,你救了允禮的孩子。」的時候,寧嬪的第一個反應不是「幹你媽妳還跟他幹過」這之類的怨懟,而是「我當年差點害這兩個孩子無法降生」..."
6,3364,67,0.550,寧貴人那句天下出家人為妳羞愧而死真的超霸氣 衝到皇上旁邊蘇培盛還讓開😂
7,3195,72,0.603,"陵容的台詞安排得很妙,每句都像在幫甄環辯護,但每句都能讓祺貴人和皇后有新話題接。"
8,2890,56,0.505,延禧攻略跟如㦤還有甄嬛 三部比一比 還是真心覺得追甄嬛好
9,2615,113,0.521,"全程無尿點,太精彩了 過了這麼多年,甄嬛的扮相還是一樣讓人感覺非常美麗,一點都沒有因為時代前進而褪色TTT"
